# Config

In [7]:
from pathlib import Path
import torch

# ===== Paths =====
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
INDEX_DIR = BASE_DIR / "index_store"
OUTPUT_DIR = BASE_DIR / "outputs"

CORPUS_PATH = DATA_DIR / "corpus.jsonl"
BENCHMARK_PATH = DATA_DIR / "benchmark.json"
TEST_QUERIES_PATH = DATA_DIR / "test_queries.json"

INDEX_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== Models =====
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# ===== Baseline settings =====
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 5
RETRIEVAL_CANDIDATE_K = 12
RETRIEVAL_ALPHA = 0.75
MAX_NEW_TOKENS = 128

# ===== Runtime =====
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available(): 
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Initialize System: Running on {DEVICE.upper()} mode")

TEMPERATURE = 0.0

# ===== Files =====
FAISS_INDEX_PATH = str(INDEX_DIR / "faiss_index")
INFERENCE_OUTPUT_PATH = OUTPUT_DIR / "inference_output.json"
EVAL_OUTPUT_PATH = OUTPUT_DIR / "evaluation_results.json"

Initialize System: Running on CUDA mode


# Indexing

In [12]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from config import (
    CORPUS_PATH,
    EMBEDDING_MODEL_NAME,
    FAISS_INDEX_PATH,
)
from utils import load_jsonl

def indexing()-> None:
    corpus = load_jsonl(CORPUS_PATH)

    documents = []
    for row in corpus:
        chunk_id = row["doc_id"]
        title = row.get("title", "")
        source = row.get("source", "")
        document_type = row.get("document_type", "")
        text = row["text"]

        enriched_chunk = f"Title: {title}\nSource: {source}\nType: {document_type}\n\n{text}"
        documents.append(
            Document(
                page_content=enriched_chunk,
                metadata={
                    "doc_id": chunk_id, 
                    "title": title,
                    "source": source,
                    "document_type": document_type,
                    "chunk_id": chunk_id,
                },
            )
        )

    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_NAME,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

    vectorstore = FAISS.from_documents(documents, embeddings)
    vectorstore.save_local(FAISS_INDEX_PATH)

    print(f"Built FAISS index with {len(documents)} chunks.")
    print(f"Saved to: {FAISS_INDEX_PATH}")

indexing()
    

Built FAISS index with 5382 chunks.
Saved to: D:\Manchester\Text-Into-Meaning-RAG\index_store\faiss_index


# utils

In [13]:
import json
import re
from pathlib import Path
from typing import List, Dict, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
from config import CHUNK_SIZE, CHUNK_OVERLAP


def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text


def exact_match(pred: str, gold: str) -> int:
    return int(normalize_text(pred) == normalize_text(gold))


def token_f1(pred: str, gold: str) -> float:
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()

    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0

    pred_counts = {}
    for t in pred_tokens:
        pred_counts[t] = pred_counts.get(t, 0) + 1

    gold_counts = {}
    for t in gold_tokens:
        gold_counts[t] = gold_counts.get(t, 0) + 1

    common = 0
    for token, count in pred_counts.items():
        if token in gold_counts:
            common += min(count, gold_counts[token])

    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def lexical_overlap_score(query: str, text: str) -> float:
    query_tokens = set(normalize_text(query).split())
    if not query_tokens:
        return 0.0
    text_tokens = set(normalize_text(text).split())
    if not text_tokens:
        return 0.0
    return len(query_tokens.intersection(text_tokens)) / len(query_tokens)


def reciprocal_rank(retrieved_doc_ids: List[str], gold_doc_ids: set[str]) -> float:
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id in gold_doc_ids:
            return 1.0 / rank
    return 0.0


def build_prompt(query: str, retrieved_chunks: List[str]) -> str:
    context = "\n\n".join(
        [f"[Chunk {i+1}]\n{chunk}" for i, chunk in enumerate(retrieved_chunks)]
    )
    return f"""You are a culinary question-answering assistant specialised in cuisine.
Use only the provided context to answer.
If the answer is not in the context, output exactly: "I cannot determine this from the provided context."
Keep the answer concise (1-2 sentences), factual, and avoid adding assumptions.

Context:
{context}

Question:
{query}

Final Answer:"""

def get_text_splitter():
    """Returns a unified text splitter for both data preparation and indexing."""
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

# Inference

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from config import (
    TEST_QUERIES_PATH,
    EMBEDDING_MODEL_NAME,
    GENERATION_MODEL_NAME,
    FAISS_INDEX_PATH,
    TOP_K,
    RETRIEVAL_CANDIDATE_K,
    RETRIEVAL_ALPHA,
    MAX_NEW_TOKENS,
    DEVICE,
    TEMPERATURE,
    INFERENCE_OUTPUT_PATH,
)
from utils import load_json, save_json, build_prompt, lexical_overlap_score


def load_vectorstore():
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_NAME,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    return FAISS.load_local(
        FAISS_INDEX_PATH,
        embeddings,
        allow_dangerous_deserialization=True,
    )


def load_generator():
    tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        GENERATION_MODEL_NAME,
        torch_dtype=torch.float32 if DEVICE == "cpu" else torch.float16,
    )
    if DEVICE == "cuda":
        model = model.to("cuda")
    model.eval()
    return tokenizer, model


def generate_answer(tokenizer, model, prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False if TEMPERATURE == 0.0 else True,
            temperature=TEMPERATURE,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    if "Final Answer:" in decoded:
        return decoded.split("Final Answer:", 1)[-1].strip()
    if "Answer:" in decoded:
        return decoded.split("Answer:", 1)[-1].strip()
    return decoded.strip().split("\n")[0]


def hybrid_retrieve(vectorstore, query: str):
    # 1) semantic recall
    docs_with_scores = vectorstore.similarity_search_with_score(
        query, k=RETRIEVAL_CANDIDATE_K
    )
    if not docs_with_scores:
        return []

    # 2) score normalization and lexical re-ranking
    max_dist = max(score for _, score in docs_with_scores)
    min_dist = min(score for _, score in docs_with_scores)

    reranked = []
    for doc, dist in docs_with_scores:
        if max_dist == min_dist:
            semantic = 1.0
        else:
            semantic = 1.0 - ((dist - min_dist) / (max_dist - min_dist))
        lexical = lexical_overlap_score(query, doc.page_content)
        hybrid = RETRIEVAL_ALPHA * semantic + (1.0 - RETRIEVAL_ALPHA) * lexical
        reranked.append((doc, hybrid, semantic, lexical))

    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked[:TOP_K]


def main() -> None:
    queries = load_json(TEST_QUERIES_PATH)
    vectorstore = load_vectorstore()
    tokenizer, model = load_generator()

    outputs = []

    for item in queries:
        qid = item["id"]
        query = item["query"]

        reranked = hybrid_retrieve(vectorstore, query)
        docs = [x[0] for x in reranked]
        retrieved_chunks = [d.page_content for d in docs]

        prompt = build_prompt(query, retrieved_chunks)
        answer = generate_answer(tokenizer, model, prompt)

        outputs.append(
            {
                "id": qid,
                "query": query,
                "answer": answer,
                "retrieved_chunks": [
                    {
                        "text": d.page_content,
                        "metadata": d.metadata,
                        "hybrid_score": round(reranked[i][1], 4),
                        "semantic_score": round(reranked[i][2], 4),
                        "lexical_score": round(reranked[i][3], 4),
                    }
                    for i, d in enumerate(docs)
                ],
            }
        )

        print(f"[{qid}] {query}")
        print(f"Answer: {answer}")
        print("-" * 60)

    save_json(outputs, INFERENCE_OUTPUT_PATH)
    print(f"Saved inference outputs to {INFERENCE_OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
c:\Users\98381\miniconda3\envs\rag\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\98381\miniconda3\envs\rag\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\98381\miniconda3\envs\rag\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sam

[1] What are the eight major culinary traditions of Chinese cuisine?
Answer: The eight major culinary traditions of Chinese cuisine include:

1. Northern styles featuring oils and strong flavors derived from ingredients like vinegar and garlic.
2. Southern styles emphasizing freshness and light preparation.
3. Eastern and western influences blending together.
4. Regional cuisines including Cantonese, Shandong, Jiangsu, Huaiyang, Sichuan, and others.
5. The Four Great Traditions of Chinese cuisine: Chuan, Lu, Yue, and Huaiyang.
6. The Four Great Cuisines of China: Shandong, Guangdong, Fujian, and Zhejiang.
7. The Four Great C
------------------------------------------------------------
[2] Which Chinese regional cuisine is most famous for its 'málà' flavor profile?
Answer: The Chinese regional cuisine that is most famous for its 'málà' flavor profile is Shaanxi cuisine. This type of dish includes noodles, fried flatbread (da bing), and a unique flavor combination called 'mala', which re

In [ ]:
from config import BENCHMARK_PATH, INFERENCE_OUTPUT_PATH, EVAL_OUTPUT_PATH
from utils import load_json, save_json, exact_match, token_f1, reciprocal_rank


def main() -> None:
    benchmark = load_json(BENCHMARK_PATH)
    predictions = load_json(INFERENCE_OUTPUT_PATH)

    pred_map = {x["id"]: x for x in predictions}

    results = []
    retrieval_hits = []
    retrieval_recalls = []
    retrieval_precisions = []
    retrieval_mrr = []
    em_scores = []
    f1_scores = []

    for ex in benchmark:
        qid = ex["id"]
        gold_answer = ex["gold_answer"]
        gold_doc_ids = set(ex["gold_evidence_doc_ids"])

        if qid not in pred_map:
            continue

        pred = pred_map[qid]
        pred_answer = pred["answer"]

        retrieved_doc_ids_list = [chunk["metadata"]["doc_id"] for chunk in pred["retrieved_chunks"]]
        retrieved_doc_ids = set(retrieved_doc_ids_list)

        hit_at_5 = int(len(gold_doc_ids.intersection(retrieved_doc_ids)) > 0)
        recall_at_5 = len(gold_doc_ids.intersection(retrieved_doc_ids)) / len(gold_doc_ids) if gold_doc_ids else 0.0
        precision_at_5 = len(gold_doc_ids.intersection(retrieved_doc_ids)) / len(retrieved_doc_ids) if retrieved_doc_ids else 0.0
        mrr_at_5 = reciprocal_rank(retrieved_doc_ids_list, gold_doc_ids)
        em = exact_match(pred_answer, gold_answer)
        f1 = token_f1(pred_answer, gold_answer)

        retrieval_hits.append(hit_at_5)
        retrieval_recalls.append(recall_at_5)
        retrieval_precisions.append(precision_at_5)
        retrieval_mrr.append(mrr_at_5)
        em_scores.append(em)
        f1_scores.append(f1)

        results.append(
            {
                "id": qid,
                "query": ex["query"],
                "gold_answer": gold_answer,
                "pred_answer": pred_answer,
                "hit@5": hit_at_5,
                "recall@5": round(recall_at_5, 4),
                "precision@5": round(precision_at_5, 4),
                "mrr@5": round(mrr_at_5, 4),
                "exact_match": em,
                "token_f1": round(f1, 4),
            }
        )

    summary = {
        "num_examples": len(results),
        "retrieval_hit@5": round(sum(retrieval_hits) / len(retrieval_hits), 4) if retrieval_hits else 0.0,
        "retrieval_recall@5": round(sum(retrieval_recalls) / len(retrieval_recalls), 4) if retrieval_recalls else 0.0,
        "retrieval_precision@5": round(sum(retrieval_precisions) / len(retrieval_precisions), 4) if retrieval_precisions else 0.0,
        "retrieval_mrr@5": round(sum(retrieval_mrr) / len(retrieval_mrr), 4) if retrieval_mrr else 0.0,
        "generation_exact_match": round(sum(em_scores) / len(em_scores), 4) if em_scores else 0.0,
        "generation_token_f1": round(sum(f1_scores) / len(f1_scores), 4) if f1_scores else 0.0,
        "details": results,
    }

    save_json(summary, EVAL_OUTPUT_PATH)
    print("Evaluation summary:")
    print(summary)
    print(f"Saved evaluation results to {EVAL_OUTPUT_PATH}")


if __name__ == "__main__":
    main()